# Train PhenX+NCIt bi-encoder — 100K subset, 1 epoch

Same pipeline as `scale20_ep025`, with the HF-blog / Alex-style subset:
`train.select(range(100_000))`, eval/test `select(range(20_000))` — **after a seeded shuffle**,
because the split CSVs store NCIt rows first and PhenX rows last (GroupShuffleSplit returns
ascending indices), so an unshuffled `range()` subset would be 100% NCIt.

Epoch budget: `num_train_epochs=1` (~1.56K steps at bs 64). The 3-epoch sweep showed hybrid Top-1
peaking near step 1,400 (~0.9 epoch) and flat thereafter, so 1 epoch reaches the same hybrid
plateau at a third of the compute. Checkpoints are saved every 200 steps for a later sweep;
best model is still selected by `eval_loss` (training signal only).

In [1]:
import os
import torch
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from peft import LoraConfig, TaskType

SEED = 42

MODEL_NAME = "uc-ctds/bge-large-en-v1.5-bio-mapping"
DATA_DIR   = "/home/fan1/notebooks/single_stage_phenx_ncit_data_0818"   # 甲-format splits from merge_and_split (0818)
OUTPUT_DIR = "/opt/gpudata/fan1/heal_cde/ncit/experiments/single_stage_phenx_ncit_scale20_subset500k_ep1_0818"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("output ->", OUTPUT_DIR)

output -> /opt/gpudata/fan1/heal_cde/ncit/experiments/single_stage_phenx_ncit_scale20_subset500k_ep1_0818


/tmp/ipykernel_2260281/3471529740.py:9: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
/tmp/ipykernel_2260281/3471529740.py:10: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


## Load data with the standard `load_dataset`

The CSV has columns `variable_para` (anchor) and `alternate_variable_para` (positive). MNRL
expects the columns in (anchor, positive) order, so we rename and drop any other columns.

In [2]:
# Load the train/val/test splits (standard HF datasets loader)
dataset = load_dataset(
    "csv",
    data_files={
        "train": os.path.join(DATA_DIR, "train.csv"),
        "eval":  os.path.join(DATA_DIR, "val.csv"),
        "test":  os.path.join(DATA_DIR, "test.csv"),
    },
)

# Rename to the standard (anchor, positive) names; KEEP `source` for now — we need it for the
# subset sanity check below. It is dropped right before training.
def to_anchor_positive_with_source(ds):
    ds = ds.rename_columns({"variable_para": "anchor", "alternate_variable_para": "positive"})
    keep = ["anchor", "positive", "source"]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    return ds

train_dataset = to_anchor_positive_with_source(dataset["train"])
eval_dataset  = to_anchor_positive_with_source(dataset["eval"])
test_dataset  = to_anchor_positive_with_source(dataset["test"])

print("train:", len(train_dataset), "| eval:", len(eval_dataset), "| test:", len(test_dataset))
print("columns:", train_dataset.column_names)
print("example anchor  :", train_dataset[0]["anchor"][:150])
print("example positive:", train_dataset[0]["positive"][:150])

train: 1114017 | eval: 138670 | test: 139018
columns: ['anchor', 'positive', 'source']
example anchor  : name: PERCUTANEOUS CORONARY INTERVENTION (PCI) FOR ST ELEVATION MYOCARDIAL INFARCTION (STEMI) (STABLE, >12 HRS FROM SYMPTOM ONSET) | description: A pe
example positive: name: Percutaneous Coronary Intervention for ST Elevation Myocardial Infarction-Stable-Over 12 Hours From Symptom Onset | description: A percutaneous 


## Shuffle + subset (Alex / HF-blog style), then sanity-check coverage

`select(range(n))` alone would take the first *n* physical rows = all NCIt.
So: seeded `shuffle` -> `select` -> verify the ncit/phenx mix against the full-split reference.

In [3]:
# subset like Alex's dataset["eval"].select(range(20_000)) — but shuffled first
TRAIN_SUBSET, EVAL_SUBSET, TEST_SUBSET = 500000, 20_000, 20_000

def report_mix(name, ds, ref_ds=None):
    n = len(ds)
    n_phenx = sum(1 for s in ds["source"] if s == "phenx")
    n_ncit = n - n_phenx
    line = f"  {name:<16}: ncit {n_ncit:>9,} ({n_ncit/n*100:.3f}%) | phenx {n_phenx:>7,} ({n_phenx/n*100:.3f}%)"
    if ref_ds is not None:
        rn = len(ref_ds)
        rp = sum(1 for s in ref_ds["source"] if s == "phenx")
        line += f"   [full split: phenx {rp/rn*100:.3f}%]"
    print(line)
    return n_phenx

full_train, full_eval, full_test = train_dataset, eval_dataset, test_dataset

train_dataset = full_train.shuffle(seed=SEED).select(range(TRAIN_SUBSET)).flatten_indices()
eval_dataset  = full_eval.shuffle(seed=SEED).select(range(EVAL_SUBSET)).flatten_indices()
test_dataset  = full_test.shuffle(seed=SEED).select(range(TEST_SUBSET)).flatten_indices()

print("subset sizes -> train:", len(train_dataset), "| eval:", len(eval_dataset), "| test:", len(test_dataset))
print("\nsource mix, subset vs full split (expect subsets ~= their full-split reference, ~1.5% phenx):")
p_tr = report_mix("train subset", train_dataset, full_train)
p_ev = report_mix("eval subset",  eval_dataset,  full_eval)
p_te = report_mix("test subset",  test_dataset,  full_test)

# hard guards: a shuffle failure shows up as (near-)zero phenx in a subset
assert p_tr > 0 and p_ev > 0 and p_te > 0, "a subset has ZERO phenx rows - shuffle did not happen?"

# training only needs (anchor, positive); MNRL reads columns in order
train_dataset = train_dataset.remove_columns(["source"])
eval_dataset  = eval_dataset.remove_columns(["source"])
test_dataset  = test_dataset.remove_columns(["source"])
print("\nfinal columns:", train_dataset.column_names)

Flattening the indices:   0%|          | 0/500000 [00:00<?, ? examples/s]

subset sizes -> train: 500000 | eval: 20000 | test: 20000

source mix, subset vs full split (expect subsets ~= their full-split reference, ~1.5% phenx):


  train subset    : ncit   492,217 (98.443%) | phenx   7,783 (1.557%)   [full split: phenx 1.554%]


  eval subset     : ncit    19,722 (98.610%) | phenx     278 (1.390%)   [full split: phenx 1.505%]


  test subset     : ncit    19,692 (98.460%) | phenx     308 (1.540%)   [full split: phenx 1.539%]

final columns: ['anchor', 'positive']


## Model + LoRA

In [4]:
model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 512

# LoRA config (use default params):
#   https://huggingface.co/docs/peft/en/package_reference/lora
lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
)
model.add_adapter(lora_config)

## Loss

In [5]:
# scale=20 is the Sentence Transformers default. scale=20 is temperature=0.05. 
# https://github.com/huggingface/sentence-transformers/blob/main/sentence_transformers/sentence_transformer/losses/multiple_negatives_ranking.py#L21
# https://sbert.net/docs/package_reference/sentence_transformer/losses.html
loss = MultipleNegativesRankingLoss(model, scale=20)
print(loss)

MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
    (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
    (2): Normalize({})
  )
)


## Training arguments (3 epochs on the 100K subset, dense checkpoints for a later sweep)

In [6]:
BATCH_SIZE = 64
LR = 5e-6 # 10x smaller than the default. A lower learning rate is common for LoRA fine-tuning and for adapting a base model

# ~7,812 steps/epoch at bs 64 on 500K rows over 1 epoch (Aarti: try 100K/300K/500K).
# eval/save every 200 steps -> ~39 eval points and checkpoints kept for a later sweep.
args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,        # default in HF blog https://huggingface.co/blog/train-sentence-transformers#trainer
    per_device_train_batch_size=BATCH_SIZE,  # HF blog uses 16
    per_device_eval_batch_size=BATCH_SIZE,   # HF blog uses 16
    learning_rate=LR,    # defaults to 5e-5 in https://github.com/huggingface/transformers/blob/main/src/transformers/training_args.py#L210
    warmup_ratio=0.1,  # default in HF blog https://huggingface.co/blog/train-sentence-transformers#trainer, Alex used as well
    bf16=True,        # H200 default, set to True if GPU supports
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # default in https://sbert.net/docs/sentence_transformer/training_overview.html
    eval_strategy="steps",    # default in https://sbert.net/docs/sentence_transformer/training_overview.html
    eval_steps=200,          
    save_strategy="steps",
    save_steps=200,           # keep every checkpoint for the post-hoc sweep
    save_total_limit=None,    # do NOT rotate old checkpoints away - the sweep needs all of them
    load_best_model_at_end=True,   # default in https://sbert.net/docs/sentence_transformer/training_overview.html
    metric_for_best_model="eval_loss",   # default in https://sbert.net/docs/sentence_transformer/training_overview.html
    greater_is_better=False,
    prediction_loss_only=True,
    eval_on_start=True,
    logging_steps=100,    # default in HF blog https://huggingface.co/blog/train-sentence-transformers#trainer
    seed=SEED,
    report_to="none",
)
print("epochs (max):", args.num_train_epochs, "| batch:", BATCH_SIZE, "| lr:", LR, "| scale: 20")
print("eval/save every", args.eval_steps, "steps | save_total_limit:", args.save_total_limit)

epochs (max): 1 | batch: 64 | lr: 5e-06 | scale: 20
eval/save every 200 steps | save_total_limit: None


## Train

In [7]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
0,No log,0.288152
200,0.375000,0.285628
400,0.348300,0.269366
600,0.291600,0.235833
800,0.256600,0.200473
1000,0.211100,0.176635
1200,0.181300,0.161554
1400,0.172000,0.152069
1600,0.154000,0.145337
1800,0.153100,0.140606


TrainOutput(global_step=7813, training_loss=0.15737689977172514, metrics={'train_runtime': 9579.5601, 'train_samples_per_second': 52.194, 'train_steps_per_second': 0.816, 'total_flos': 0.0, 'train_loss': 0.15737689977172514, 'epoch': 1.0})

## Evaluate on the held-out test set

Final number for hyperparameter decisions. The HEAL benchmark stays untouched.

In [8]:
test_metrics = trainer.evaluate(test_dataset, metric_key_prefix="test")
print("test-set metrics:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

test-set metrics:
  test_loss: 0.1131218746304512
  test_runtime: 93.344
  test_samples_per_second: 214.261
  test_steps_per_second: 3.353
  epoch: 1.0


## Save the best model

`load_best_model_at_end=True` already reloaded the lowest-eval_loss checkpoint; save it as `final/`.

In [9]:
final_dir = os.path.join(OUTPUT_DIR, "final")
model.save_pretrained(final_dir)
print("saved best model ->", final_dir)

saved best model -> /opt/gpudata/fan1/heal_cde/ncit/experiments/single_stage_phenx_ncit_scale20_subset500k_ep1_0818/final
